# GeneVariate — Google Colab runner 🧬

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SciSpectator/genevariate/blob/main/notebooks/genevariate_colab.ipynb)

Runs the **GPU/VRAM-heavy** parts of GeneVariate on Colab's GPU:

* the **LLM-GEO label extractor** pipeline (`genevariate-llm-extract`), backed by a local **Ollama** model, and
* the **BioLORD-2023** semantic curator (torch / sentence-transformers), and
* the novel **analysis API** (ΔVariance GSEA, bimodality-gated, meta-enrichment, pseudo-cohorts).

> **The desktop GUI (`genevariate`) is tkinter-based and cannot render on Colab's headless runtime.** Colab is used here for compute — the LLM pipeline, the analysis functions, and the test suite. Run the GUI on your local device (`pip install -e ".[analysis]"` then `genevariate`).

### ⚙️ First: switch on the GPU
**Runtime → Change runtime type → Hardware accelerator: GPU (T4 is fine; L4/A100 give more VRAM), then Save.**

## 1 · Confirm the GPU is attached

In [ ]:
!nvidia-smi || echo '⚠️  No GPU — go to Runtime → Change runtime type → GPU, then rerun.'

## 2 · Clone the newest GeneVariate
Always pulls the latest `main` from GitHub.

In [ ]:
import os, subprocess
REPO_DIR = '/content/genevariate'
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/SciSpectator/genevariate.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
os.chdir(REPO_DIR)
print('commit:', subprocess.check_output(['git','-C',REPO_DIR,'log','-1','--oneline']).decode().strip())

## 3 · Install GeneVariate + analysis + LLM-extractor deps
Colab already ships a CUDA-enabled **torch**, so BioLORD runs on the GPU. This installs the
`analysis` (GSEA / ARCHS4 / statsmodels) and `llm-extract` (dspy / sentence-transformers) extras.

In [ ]:
# torch is preinstalled on Colab GPU runtimes; don't let pip downgrade it.
!pip install -q -e ".[analysis,llm-extract]"
print('\n✅ install finished')

## 4 · Install & start Ollama (the local LLM server)
Ollama serves the extraction model on the Colab GPU. We start it in the background.

In [ ]:
import time, subprocess, requests
# Install Ollama (idempotent)
!curl -fsSL https://ollama.com/install.sh | sh
# Start the server in the background
subprocess.Popen(['ollama', 'serve'],
                 stdout=open('/content/ollama.log','w'),
                 stderr=subprocess.STDOUT,
                 env={**os.environ, 'OLLAMA_HOST':'127.0.0.1:11434'})
# Wait until it answers
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            print('✅ Ollama is up'); break
    except Exception:
        time.sleep(1)
else:
    print('⚠️  Ollama did not start — see /content/ollama.log')

## 5 · Pull the models — with tag fallback
GeneVariate's code asks for the tags **`gemma4:e2b`** and **`gemma4-e2b-text:latest`**. Those aren't
in the public Ollama registry (`gemma4` doesn't exist there; `e2b` is Google's **Gemma 3n E2B**
effective-2B variant). So we pull a real model and **alias** it to the exact tag names the code
expects with `ollama cp` — no source edits needed. `nomic-embed-text` is pulled for embeddings.

In [ ]:
import subprocess

def sh(cmd):
    print('$', cmd)
    return subprocess.run(cmd, shell=True).returncode

def have(tag):
    out = subprocess.run(['ollama','list'], capture_output=True, text=True).stdout
    return tag.split(':')[0] in out and (tag in out or ':latest' in out)

WANT = ['gemma4:e2b', 'gemma4-e2b-text:latest']

# 1) Try the exact tags the repo references (works if you have a private/registry copy).
base = None
if sh('ollama pull gemma4:e2b') == 0:
    base = 'gemma4:e2b'
else:
    # 2) Fallback: pull the real upstream Gemma 3n E2B and alias it.
    for cand in ['gemma3n:e2b', 'gemma3:4b', 'gemma2:2b']:
        if sh(f'ollama pull {cand}') == 0:
            base = cand; break

assert base, 'Could not pull any Gemma model — check /content/ollama.log'
print('base model =', base)

# Alias the base model to every tag the code expects.
for tag in WANT:
    if base != tag:
        sh(f'ollama cp {base} {tag}')

# Embeddings model (semantic RAG / pseudo-cohorts)
sh('ollama pull nomic-embed-text')

print('\n=== ollama list ===')
!ollama list

## 6 · Smoke test A — the analysis API (no LLM needed)
Exercises the novel enrichment / distribution code on a tiny synthetic matrix so you know the
package imports and computes correctly before spending GPU time.

In [ ]:
import numpy as np, pandas as pd
from genevariate.core.analysis import (
    rank_genes_by_variability, classify_distributions, filter_ranked_by_distribution,
)

rng = np.random.default_rng(0)
genes = [f'GENE{i}' for i in range(50)]
samples = [f'S{i}' for i in range(20)]
df = pd.DataFrame(rng.normal(size=(20, 50)), index=samples, columns=genes)
labels = pd.Series(['case']*10 + ['ctrl']*10, index=samples)
# make a few genes genuinely more variable in cases
df.loc[labels=='case', genes[:5]] *= 4.0

ranked = rank_genes_by_variability(df, labels, 'case', 'ctrl', method='logvar_z')
tags = classify_distributions(df)
print('Top ΔVariance genes:')
print(ranked.head(8))
print('\nDistribution tags (sample):')
print(dict(list(tags.items())[:8]))
print('\n✅ analysis API works')

## 7 · Smoke test B — the LLM on the GPU
Directly calls the aliased Ollama model to prove inference runs on VRAM, then shows the
extractor CLI is wired up.

In [ ]:
import ollama
r = ollama.chat(model='gemma4:e2b',
                messages=[{'role':'user',
                           'content':'In one word, what tissue is "peripheral blood mononuclear cells"?'}])
print('LLM says:', r['message']['content'].strip()[:200])
print('\n--- extractor CLI help ---')
!genevariate-llm-extract --help | head -40

## 8 · (Optional) Run a real extraction
Runs the full pipeline on a small GEO series. `--scrape` fetches metadata from NCBI, so no
multi-GB `GEOmetadb.sqlite` download is required for a single-series run. Adjust `--gpl` /
`--limit` as you like; results land in `--output`.

In [ ]:
# Example: cap to a handful of samples so it finishes quickly on a T4.
# Replace GPL570 with your platform of interest.
!genevariate-llm-extract \
    --phases all \
    --gpl GPL570 \
    --limit 5 \
    --workers 2 \
    --output /content/extract_out \
    --model gemma4-e2b-text:latest || echo '(adjust flags — see --help above)'
print('\nOutput dir:'); !ls -la /content/extract_out 2>/dev/null || true

## 9 · (Optional) Run the test suite
Validates every `core/analysis` module and the cross-technology loaders on the GPU box.

In [ ]:
!pip install -q pytest
!python -m pytest tests/ -q -x -k 'variability or enrichment or bimodality or pseudo or meta' || true

---
### Notes
* **GUI:** `genevariate` (the 3-step tkinter desktop app) needs a display — run it locally, not on Colab.
  If you truly need the GUI in Colab you'd have to tunnel it over VNC/noVNC, which is fragile; the
  headless pipeline above is the intended Colab workload.
* **Persisting results:** mount Drive (`from google.colab import drive; drive.mount('/content/drive')`)
  and point `--output` there so extractions survive the runtime being recycled.
* **More VRAM:** T4 (16 GB) handles Gemma 3n E2B + BioLORD comfortably. For bigger models pick
  L4 / A100 under Runtime → Change runtime type.